# Project 05: Sephora Products and Reviews

Project by Daniel Perez

### Background

This project utilizes the [Sephora Products and Skincare Reviews](https://www.kaggle.com/datasets/nadyinky/sephora-products-and-skincare-reviews?select=product_info.csv) datasets provided by Nady Inky on Kaggle.<br>One dataset contains data on around 8 thousand products from Sephora's website, while the other six datasets contain data on around 1.3 million reviews of skincare products purchased on the site.

### Table of Contents

- [Overview](#Overview)
- [Data Cleaning](#data_cleaning)
- [Exploratory Data Analysis](#exploratory_data_analysis)
- [Sentiment Analysis](#sentiment_analysis)
- [Sentiment Analysis Model](#building_the_sentiment_analysis_model)
- [Visualization](#visualization)
- [Key Takeaways and What's Next](#key_takeaways_and_whats_next)

### Overview <a id='Overview'></a>

In this project, we will use Python to answer the following business tasks:<br>
1) __Exploratory Data Analysis:__ Explore product categories, brand popularity, and products with similar ingredients.<br>
<br>
2) __Sentiment Analysis:__ Is the emotional tone of the review positive or negative? Which brands or products have the most positive or negative reviews?
<br>
<br>
3) __Text Analysis:__ What do customers say most often in their positive and negative reviews? Do customers have any common problems with their skincare?
<br>
<br>
4) __Data Visualization:__ What are the most popular brands and products? What is the distribution of prices? Which products are closest to each other in ingredients? What does the word cloud of the most frequently used words look like?
<br>
<br>

Our first step in answering to these tasks is to load in the data we downloaded locally and preview the dataframe.<br>We can do this with the following Python code:

In [ ]:
# importing necessary libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# loading in products data

data = pd.read_csv("/kaggle/input/sephora-products-and-skincare-reviews/product_info.csv")
data.info(verbose=True)

From this, we gather that the products dataset contains 27 columns and 8,494 rows of data. The columns vary in data types, and there is missing data within the dataframe.<br>

For the datasets related to the reviews, it will be helpful to join them all into one dataframe, which we can do with the following:

In [ ]:
# getting the files

t1 = pd.read_csv('/kaggle/input/sephora-products-and-skincare-reviews/reviews_0-250.csv', low_memory=False)
t2 = pd.read_csv("/kaggle/input/sephora-products-and-skincare-reviews/reviews_250-500.csv", low_memory=False)
t3 = pd.read_csv("/kaggle/input/sephora-products-and-skincare-reviews/reviews_500-750.csv", low_memory=False)
t4 = pd.read_csv("/kaggle/input/sephora-products-and-skincare-reviews/reviews_750-1250.csv", low_memory=False)
t5 = pd.read_csv("/kaggle/input/sephora-products-and-skincare-reviews/reviews_1250-end.csv", low_memory=False)
# t6 = pd.read_csv("/kaggle/input/sephora-products-and-skincare-reviews/reviews_1500-end.csv", low_memory=False)

# combining the dfs

# texta = pd.concat([t1,t2,t3,t4,t5,t6])
texta = pd.concat([t1,t2,t3,t4,t5])
texta.info()

We gather that there are around 1.09 million reviews with varying amounts of missing data across 19 columns of the 6 datasets.

It can also be helpful to view the numeric and non-numeric columns of both dataframes. We can do so with the following:

#### Numeric Columns

In [ ]:
numeric_cols = data.select_dtypes(include = ['number']).columns
print(numeric_cols)
print(f'{len(numeric_cols)} Numeric Columns in Products Dataset')

In [ ]:
numeric_cols_reviews = texta.select_dtypes(include = ['number']).columns
print(numeric_cols_reviews)
print(f'{len(numeric_cols_reviews)} Numeric Columns in Reviews Dataset')

#### Non-Numeric Columns

In [ ]:
non_numeric_cols = data.select_dtypes(exclude=['number']).columns
print(non_numeric_cols)
print(f'{len(non_numeric_cols)} Non-Numeric Columns in Products Dataset')

In [ ]:
non_numeric_rev_cols = texta.select_dtypes(exclude=['number']).columns
print(non_numeric_rev_cols)
print(f'{len(non_numeric_rev_cols)} Non-Numeric Columns in Reviews Dataset')

## Data Cleaning <a id='data_cleaning'></a>

The next step in our analysis is to clean our products dataset. This includes checking the extent of missing data, removing data that is unnecessary to our analysis, checking for outliers, and reformatting if necessary.

### Missing Data

Here we use the following code to see the amount of nulls by column within the dataset:

In [ ]:
num_missing = data.isna().sum()
num_missing

This information would probably be more helpful in a percentage format, so we can easily assess whether columns with a high percentage of missing rows are necessary for the analysis.<br>We can do that with the following:

In [ ]:
pct_missing = data.isna().mean()
pct_missing

Immediately we notice a few columns with a large percent of values missing. We can visualize just how much of the rows are missing by using Seaborn, a python visualization library.

In [ ]:
# heatmap to visualize missing data (products)

plt.figure(figsize=(10,8))

cols= data.columns
colors=['#d1d1e0','#cc0000']
sns.heatmap(data[cols].isna(),cmap=sns.color_palette(colors))

From this, we can make note of the columns with a high percentage of missing data for when we assess which columns to remove from our dataset.

We can do the same for the combined Reviews (texta) dataset. 

In [ ]:
# heatmap to visualize missing data (reviews)

plt.figure(figsize=(10,8))

cols= texta.columns
colors=['#660066','#00ffff']
sns.heatmap(texta[cols].isna(),cmap=sns.color_palette(colors))

### Outliers

The next step in our cleaning involves searching for and addressing outliers within our dataset. It is important to deal with potential outliers early in that they can alter the calculations and visualizations of grouped data later on.

We will use kurtosis (a measure of the tailedness or skew of data points relative to the center of a distribution) to check for outliers in our numeric columns. Higher kurtosis values are linked with a greater probability of outliers within the set.

In [ ]:
data.kurt(numeric_only=True)

We notice that the price_usd column representing the prices in U.S. dollars within the Products dataset has a significantly higher kurtosis value than the other numerical columns. We can use the describe() method within Python to check for an outlier on the left or right side of distribution.

In [ ]:
data['price_usd'].describe()

We notice that the max value is at 1,900 USD, while half of the data lies between 25 USD and 58 USD. To confirm our beliefs, we can visualize the column with a boxplot before extracting the outlier(s).

In [ ]:
data.boxplot(column=['price_usd'])

In [ ]:
data.loc[data['price_usd']==1900]

After doing some digging on Sephora's site, we can confirm that the price of the product above is legitimate. However, we will still exclude the outlier from the data to better gauge the price distribution later on.

We'll now check for the existence of outliers in the Reviews dataset.

In [ ]:
texta.kurt(numeric_only=True)

We notice that the columns related to the feedback counts have a high likelihood of outliers. Fortunately, these columns will not be used in the analyses of this project as they do not pertain to our business tasks.

### Unnecessary Data

The next step in cleaning will be assessing whether columns in the dataframe are necessary to our business tasks. Their exclusion may be due to:
1) Redundancy (largely repetitive columns)
2) Relevancy (whether it will aid us in our investigation)
3) Completeness (too many NaN's and nulls to be usable)

We will check for redundancy within the columns by printing out the columns with over 50% of their rows having the same value. We can also print what each column's most occurring values are to better gauge which columns provide insight.

In [ ]:
num_rows = len(data)

for col in data.columns:
    counts=data[col].value_counts(dropna=False)
    top_pct=(counts/num_rows).iloc[0]
    
    if top_pct > 0.50:
        print('{0}:{1:2f}%'.format(col,top_pct*100))
        print(counts)
        print()

From the Products dataset, we can now see the columns listed with over 50% of the same value.

- variation_desc - 85.28% NaN
- value_price_usd - 94.69% NaN
- sale_price_usd - 96.82% NaN

These three columns contain mostly NaN values, so they will be dropped from the dataframe.

- limited_edition - boolean
- new - boolean 
- online_only - boolean
- out_of_stock - boolean
- sephora_exclusive - boolean

These columns all have a boolean data type (True/False), so they will not be excluded for redundancy. However, these columns provide no insight towards our business tasks, so they will be dropped from the dataframe on account of relevancy.

- child_count - 67.57% 0 children
- child_max_price - 67.57% NaN
- child_min_price - 67.57% NaN

The child_count column is repetitive in nature, the child_max_price and child_min_price columns are largely NaN's, and these columns are not relevant to our business tasks. Therefore, we will drop them from the final Products dataframe.

In [ ]:
# we will use the following code to drop the columns and create a new df

data_edited = data.drop(columns = ['variation_desc',
                                   'value_price_usd',
                                   'sale_price_usd',
                                   'limited_edition',
                                   'new','online_only',
                                   'out_of_stock',
                                   'sephora_exclusive',
                                   'child_count',
                                   'child_max_price',
                                   'child_min_price'],axis = 1)

# while we're at it, we can remove the outlier we discovered earlier

data_edited = data_edited[data_edited.price_usd != 1900]

data_edited.info()

The new Products dataframe, data_edited, now contains 16 of the 27 columns, which gives us a lot less missing data to work with and a slightly smaller dataset to aid in calculations and processing.

Let's see if we can do the same for the Reviews dataset.

In [ ]:
num_rows_revs = len(texta)

for col in texta.columns:
    counts=texta[col].value_counts(dropna=False)
    top_pct=(counts/num_rows_revs).iloc[0]
    
    if top_pct > 0.50:
        print('{0}:{1:2f}%'.format(col,top_pct*100))
        print(counts)
        print()

As mention previously, the columns concerned with feedback counts can be dropped on account of relevancy. The helpfulness columns is largely NaN, and the is_recommended boolean column is not relevant. Although the columns concerned with the features of users could be useful in future analyses, it will not serve us in our sentiment and text analyses.

To drop these columns and create a new dataframe, we can perfom the following code:

In [ ]:
text_edited = texta.drop(columns = ['is_recommended',
                                    'helpfulness',
                                    'total_feedback_count',
                                    'total_neg_feedback_count',
                                    'total_pos_feedback_count',
                                    'skin_tone',
                                    'eye_color',
                                    'skin_type',
                                    'hair_color'])
text_edited.info()

## Exploratory Data Analysis <a id='exploratory_data_analysis'></a>

In this section, we will explore product categories and ingredient trends. We will explore brand popularity after covering sentiment analysis.

### Product Categories

In [ ]:
data_edited.primary_category.value_counts()

The results above tell us that the Products data gathered is divided into nine "primary" categories, from which the products are further filtered by "secondary" and "tertiary" categories. A large percentage of the data is composed of products that fall under the skincare category, which may be useful to us when we complete the sentiment and text analyses of the skincare reviews dataset.

In [ ]:
data_edited.secondary_category.value_counts()

Our secondary category results show us a more lengthened list compared to the primary categories, with 41 categories compared to our original 9.

In [ ]:
data_edited.tertiary_category.value_counts()

As expected, the length of distinct categories drastically increases as we refine our search, with 118 under tertiary.

### Most Similar Products by Ingredients Using Cosine Similarity

One method we can utilize to find products that are most similar to each other is Cosine Similarity. In this method, we can find the similarity of two documents regardless of difference in size by measuring the cosine of the angle between two vectors in a matrix.<br> We first convert the text strings (our ingredients lists) to word vectors in a matrix. We then find the angle between vectors in the matrix and generate a score from 0 to 1, with values closer to 0 showing less similarity and values closer to 1 showing more similarity.

We will first create a new dataframe with only the columns we are interested in.

In [ ]:
ing = pd.DataFrame(data_edited, columns=['product_id','product_name','brand_name','ingredients','price_usd'])

Then, we will remove the "sets" from the dataframe, since we are interested in comparing single products to each other and not products with nested elements.

In [ ]:
ing = ing[ing['ingredients'].str.split(':').str.len()<2]

Next, we remove products that do not have ingredients listed, and reset the index of the final dataframe. Resetting the index is crucial to matching the indices of products to the most similar product later on.

In [ ]:
# dropping products with no ingredients

ing = ing.dropna()

# restting the index

ing = ing.reset_index(drop=True)

#printing the resulting shape of the dataframe, which is 6,067 rows by 5 columns

ing.shape

We will be using the TfidfVectorizer from Python's sklearn module for this task. TF-IDF (term frequency - inverse document frequency) is a numerical statistic that reflects how significant a certain word is to a document. We can use TfidfVectorizer to create vectors of the lists of ingredients.

In [ ]:
# necessary imports

from sklearn.feature_extraction.text import TfidfVectorizer

# extracting the values from the ingredients column as our corpus
texts = ing.ingredients.values

tfidf = TfidfVectorizer().fit_transform(texts)

# vectorizer automatically returns a normalized tf-idf

pairwise_similarity = tfidf * tfidf.T

In [ ]:
pairwise_similarity

This returns a 6067x6067 sparse matrix composed of mainly zeros. We will convert this to an array to allow us to work with it with the numpy module.

In [ ]:
pairwise_similarity.toarray()

We can find the index of the most similar text string (list of ingredients) by taking the argmax of each row. This will return a list of indices which we can use to iterate through our existing products dataframe.<br> Firstly, we will need to mask the 1 values within the array as NaN values, as the 1 values indicate a product's similarity in relation to itself.

In [ ]:
arr = pairwise_similarity.toarray()
np.fill_diagonal(arr, np.nan)
arr

In [ ]:
maxes = np.nanargmax(arr, axis=0)

maxes.shape

By taking the argmax, we now have an array of the indices of the most similar products in relation to the row they reside in. We see that the shape of the resulting array is 6,067 rows by 1 column. We will convert the array to a dataframe object to work with it within the Pandas library.

In [ ]:
# converting the array to a dataframe object

ast = pd.DataFrame(maxes)
ast.shape

In [ ]:
# creating the "most similar index" column from our new dataframe object and appending it to our products dataset

ing['most_sim_index'] = ast
ing.head()

For our next step, we will take the values within the product_name column and assign them to a variable, "products". We will then take the column of indices, "most_sim_index", and assign it to a variable. Then, we can create a new column, "most_sim_product" with the names of the products at each of the indices by using our two new variables to iterate through each of the rows.

In [ ]:
products = ing.product_name.values

idxes = ing['most_sim_index']

ing['most_sim_product'] = products[idxes]
ing.head()

We can see that the indices column iteration worked as intended, though some of the results are returning products that are simply the travel size variation of the same product, which is not very useful to us. In order to remove these from the results, we will need to modify our original dataframe, which will therefore require us to restart the whole process.

In [ ]:
ing2 = ing.drop('most_sim_index', axis=1)
ing2 = ing2.drop('most_sim_product', axis=1)
ing2 = ing2[~ing2['product_name'].str.contains('Travel')]
ing2 = ing2[~ing2['product_name'].str.contains('travel')]
ing2.shape

After removing the columns appended in the last steps and the rows that contain the word "Travel" or "travel", our resulting dataframe's shape is 5,833 rows by 5 columns, which is a difference of 234 rows.

In [ ]:
ing2 = ing2.reset_index(drop=True)

texts2 = ing2.ingredients.values
tfidf2 = TfidfVectorizer().fit_transform(texts2)

pairwise_similarity2 = tfidf2 * tfidf2.T
pairwise_similarity2.toarray()

arr2 = pairwise_similarity2.toarray()
np.fill_diagonal(arr2, np.nan)

maxes2 = np.nanargmax(arr2, axis=0)

ast2 = pd.DataFrame(maxes2)

ing2['most_sim_index'] = ast2

products2 = ing2.product_name.values
idxes2 = ing2['most_sim_index']

ing2['most_sim_product'] = products2[idxes2]
ing2.head()

Finally, we have a new dataframe that provides us with the most similar product based on the cosine similarity of the ingredients.

We can build upon this by using the same technique we used with iterating the indices to now fetch the brands and prices of the most similar products as well, as this information could be valuable to us.

In [ ]:
prices = ing2.price_usd.values
brands = ing2.brand_name.values

ing2['price_sim'] = prices[idxes2]
ing2['brand_sim'] = brands[idxes2]

ing2.head(15)

This dataframe, combined with both a sentiment and sales dataframe, would allow us to analyze both the sales and sentiment performance of each (applicable) top product. We would would then be able to compare their performances against their acquisition costs and profitability, and determine whether their most similar counterpart should be advertised to customers instead.

## Sentiment Analysis <a id='sentiment_analysis'></a>

We would like to know whether the emotional tone of reviews are positive or negative, as well as the products and brands that are rated the best/worst. For this task, we will utilize sentiment analysis in order to get our answers.

In [ ]:
text_edited.head()

We already have the columns that we need: rating (a score from 1-5 with 5 being the best) and review_text (the entirety of a review's text)<br> We will look at the overall score distribution to get an idea of what to expect.

In [ ]:
# imports 

color = sns.color_palette()
%matplotlib inline
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.tools as tls
import plotly.express as px

# produce scores

fig = px.histogram(text_edited, x="rating")
fig.update_traces(marker_color='maroon',marker_line_color='gray',marker_line_width = 1.5)
fig.update_layout(title_text='Product Score')
fig.show()

This tells us that most ratings are positive, which indicates that most reviews will likely be positive as well.<br>We can get an idea of what's being said the most using a wordcloud, or a visualization of the most used words in multiple texts with larger words indicating more frequent usage.

After initially running the wordcloud, I noticed a few stopwords located within the visual. Stopwords are words that are deemed insignificant in Natural Language Processing (the, and, a, to, etc.) and are thus filtered out of most analyses. To manually add stopwords, we can use the stopwords.update() function as shown below.

In [ ]:
# imports

import nltk
from wordcloud import WordCloud
from nltk.corpus import stopwords

# ensuring that the reviews and titles are in fact string datatypes

text_edited['review_text'] = text_edited['review_text'].astype(str)
text_edited['review_title'] = text_edited['review_title'].astype(str)

# create stopword list

stopwords = set(stopwords.words())
stopwords.update(['day','night','received','make','week','morning','put','leave'])

# generating a wordcloud and plotting the results

texxt = " ".join(review_title for review_title in text_edited.review_text)
wordcloud = WordCloud(stopwords=stopwords).generate(texxt)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

The general wordcloud can provide some insight, but what would be even more insightful would be to see what is primarily being said within reviews that we classify as negative and those that we classify as positive.

In order to do so, we will first need to classify the reviews by their ratings. A score of 3 is considered the neutral point between the two spectrums of our rating scale, so we will drop the reviews that contain a rating of 3.<br>We will then assign a negative value to ratings that range from 1-2 and a positive value for those that range from 4-5.

In [ ]:
textb = text_edited[text_edited['rating'] !=3]
textb['sentiment']= textb['rating'].apply(lambda rating: +1 if rating >3 else -1)
textb.head()

We can then split the existing dataframe into positive and negative to generate a wordcloud for each.

In [ ]:
# split df into positive and negative

positive = textb[textb['sentiment']==1]
negative = textb[textb['sentiment']==-1]

### Positive Wordcloud

In [ ]:
# setting stopwords for the positive wordcloud

# stopwords = set(stopwords.words())
stopwords.update(['received','day','make','feel','leave','buy','noticed','good','product','great', 'nan','work','works','stuff','finally'])

# generating a wordcloud and plotting the results

pos = " ".join(review_title for review_title in positive.review_title)
wordcloud2= WordCloud(stopwords=stopwords).generate(pos)

plt.imshow(wordcloud2, interpolation='bilinear')
plt.axis("off")
plt.show()

After some refining, the positive wordcloud allows us to paint a picture of what is being said in positive reviews:<br>
- Customers are much more prone to mentioning their skin type in relation to the product, with words like "acne prone", "dry skin", "oily skin", "soft skin", and "sensitive skin" showing within the wordcloud.<br><br>
- We see indications of newly repeating customers as a result of their experience with a product, with words like "routine", "favorite", "obsessed", "staple", "game changer", and most notably, "Holy Grail".<br><br>
- Customers have a tendency to review their specific product while mentioning the entire class of products ("serum", "sunscreen", "toner", "lip balm", "eye cream"), indicating comparisons between their history with previous products.

### Negative Wordcloud

In [ ]:
neg = " ".join(review_title for review_title in negative.review_title)
wordcloud7 = WordCloud(stopwords=stopwords).generate(neg)
stopwords.update(['work','wanted','caused','made','makes','buy','love'])

plt.imshow(wordcloud7, interpolation='bilinear')
plt.axis("off")
plt.show()

With the negative wordcloud complete, we can draw some conclusions about the negative product review consensuses:<br>
- Words concering a products effectiveness in relation to its price are immediately noticable, with words like "money", "overpriced", "worth", "waste", "buy", and "price" within the wordcloud. Customers are most upset with the investment in a product when it doesn't meet their expectations, and more so when that investment is larger.<br><br>
- Customers are prone to mentioning their skin type ("sensitive skin", "acne prone", "oily skin", etc.) in relation to a product when the review is negative.<br><br>
- Customers also mention qualities about the product itself much more frequently in negative reviews, with words like "fragrance", "formula", "packaging", "sticky", "heavy", "texture", and "scent" appearing within the wordcloud.<br><br>
- The verbs within the wordcloud illuminate some of the most common negative skin reactions to products, such as "sting", "drying", "break outs", "burn", "dries", and "irritating".

We can confirm our initial impression of the dataset by visualizing the sentiment we generated through the ratings column.

In [ ]:
# distribution of reviews by sentiment

textb['sentimentt'] = textb['sentiment'].replace({-1: 'negative'})
textb['sentimentt'] = textb['sentimentt'].replace({1: 'positive'})
fig=px.histogram(textb, x='sentimentt')
fig.update_traces(marker_color='maroon',marker_line_color='black',marker_line_width=1.5)
fig.update_layout(title_text='Product Sentiment')
fig.show()

As expected, most of the reviews are positive.

## Building the Sentiment Analysis Model<a id='building_the_sentiment_analysis_model'></a>

We can use the reviews datasets and simple logistic regression to build a classification model that predicts whether reviews are positive or negative. We will need to perform a few tasks first.

#### Removing Punctuation

In [ ]:
textb['review_text'] = textb['review_text'].astype(str)
textb['review_title'] = textb['review_title'].astype(str)

def remove_punctuation(text):
    final="".join(u for u in text if u not in ("?",".",";",":","!",'"'))
    return final

textb['review_text'] = textb['review_text'].apply(remove_punctuation)
textb=textb.dropna(subset=['review_title'])
textb['review_title']=textb['review_title'].apply(remove_punctuation)

#### Split the Dataframe

We will create a new dataframe with only two columns: review title and sentiment.

In [ ]:
# new df only two cols: review_title and sentiment

new_text = textb[['review_title','sentiment']]
new_text.head()

We will now split the dataframe into train and test sets. 80% of the data will be used for training and 20% will be used for testing.

In [ ]:
# random split train and test data

index = textb.index
textb['random_number'] =np.random.randn(len(index))

train=textb[textb['random_number']<=0.8]
test=textb[textb['random_number']>0.8]

#### Create a Bag of Words

We will now transform the text to a BoW (bag of words) model, essentially a matrix of how often each word occurs. We need to convert to a BoW model since logistic regression cannot understand text.

In [ ]:
# count vectorizer

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(token_pattern=r'\b\w+\b')

train_matrix = vectorizer.fit_transform(train['review_title'])
test_matrix = vectorizer.transform(test['review_title'])

#### Import Logistic Regression

In [ ]:
# log reg

from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

#### Split Target and Independent Variables

In [ ]:
X_train = train_matrix
X_test = test_matrix
y_train = train['sentiment']
y_test = test['sentiment']

#### Fit Model on Data

In [ ]:
lr.fit(X_train,y_train)

In [ ]:
predictions = lr.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
new=np.asarray(y_test)
confusion_matrix(predictions,y_test)

The confusion matrix we obtain after testing tells us the following:<br>
- There were 16,351 true positives (guessed positives that were actually positive)
- There were 2,734 false positives (guessed positives that were actually negative)
- There were 12,725 false negatives (guessed negative and were actually positive)
- There were 223,659 true negatives (guessed negative and were actually negative)

In [ ]:
print(classification_report(predictions,y_test))

From the classification report, we see that we obtained an overall accuracy of 94% without any feature extraction or much preprocessing with our model.<br> This model could ideally be refined and implemented into future data influx for classification purposes. For instance, Sephora can offer a different brand's coupon code for users that the model predicted to have a negative review of a product, and a coupon code for a similar product for those it classified as positive.

## Visualization<a id='visualization'></a>

For visualization, we will need to summarize and get the top 10 performers between brands and products with regards to loves_count (the number of people who have marked the product as a favorite) and rating (the average rating of the product).

In [ ]:
print(data_edited['product_name'].nunique())
print(data_edited['brand_name'].nunique())

In the products dataset, we have 8,414 unique products across 304 distinct brands.

In [ ]:
data_edited.groupby('brand_name')['loves_count'].sum().nlargest(10)

From this we gather that the SEPHORA COLLECTION brand is leading the way in loves_count, with 1.25 million across the site.

In [ ]:
sns.set(rc={'figure.figsize':(16,9)})

k = data_edited.groupby('brand_name', as_index=False)['loves_count'].sum().sort_values(by='loves_count', ascending=False).head(5)
sns.barplot(data = k,
            x = 'brand_name',
            y = 'loves_count',
            hue = 'brand_name',
            dodge = False).set(xticklabels=[])

With the visual we see how dramatic the difference is between first and second place in terms of brands, with a +3 million loves difference.

In [ ]:
data_edited.groupby('brand_name')['loves_count'].sum().nsmallest(5)

The Maker brand is Sephora's least 'loved' across the roughly 300 brands on the site.

In [ ]:
sns.set(rc={'figure.figsize':(16,9)})

k = data_edited.groupby('brand_name', as_index=False)['loves_count'].sum().sort_values(by='loves_count', ascending=True).head(5)
sns.barplot(data = k,
            x = 'brand_name',
            y = 'loves_count',
            hue = 'brand_name',
            dodge = False).set(xticklabels=[])

And from the visual, we see just how little loves the bottom 5 performers have accumulated.

After visualizing, I noticed that the ratings are not very insightful by brand, as most brands fall somewhere between 4 and 5 when averaged.<br>Similarly, the top/bottom products are at both ends of the 1-5 spectrum, with little noticable difference between top performers.

In [ ]:
sns.set(rc={'figure.figsize':(16,9)})

k = textb.groupby('brand_name', as_index=False)['sentiment'].sum().sort_values(by='sentiment', ascending=False).head(5)
sns.barplot(data = k,
            x = 'brand_name',
            y = 'sentiment',
            hue = 'brand_name',
            dodge = False).set(xticklabels=[])

If we visualize the sum of our generated 'sentiment' column, we get the difference between all negative and positive reviews under the brand. This leaves us with CLINQUE in the lead, with a difference of over positive 40,000.

In [ ]:
sns.set(rc={'figure.figsize':(16,9)})

k = textb.groupby('brand_name', as_index=False)['sentiment'].sum().sort_values(by='sentiment', ascending=True).head(5)
sns.barplot(data = k,
            x = 'brand_name',
            y = 'sentiment',
            hue = 'brand_name',
            dodge = False).set(xticklabels=[])

Generating the difference also allows us to see performers on the other extreme dip into the negatives, which we can clearly see with our worst performing brand across the site, TWEEZERMAN.

In [ ]:
sns.set(rc={'figure.figsize':(16,9)})

k = textb.groupby('product_name', as_index=False)['sentiment'].sum().sort_values(by='sentiment', ascending=False).head(5)
sns.barplot(data = k,
            x = 'product_name',
            y = 'sentiment',
            hue = 'product_name',
            dodge = False).set(xticklabels=[])

Our best product in terms of sentiment, Lip Sleeping Mask Intense Hydration with Vitamin C, drastically outperforms the other top 4, with around 11,500 positive difference compared to the roughly 6,000 positive difference garnered by the others.

In [ ]:
sns.set(rc={'figure.figsize':(16,9)})

k = textb.groupby('product_name', as_index=False)['sentiment'].sum().sort_values(by='sentiment', ascending=True).head(5)
sns.barplot(data = k,
            x = 'product_name',
            y = 'sentiment',
            hue = 'product_name',
            dodge = False).set(xticklabels=[])

The worst performing products on the site have all managed to dip into the negatives, with Clean Cleansing & Gentle Exfoliating Wipes performing the absolute worst.

### Price Distribution

#### All Products

In [ ]:
sns.displot(data_edited,
            x = "price_usd",
            hue = "primary_category",
            element = "step")

The price distribution of all products offers some insight into where most categories reside, with a bulk of the distribution keeping just under 50.00 USD. The distribution is skewed to the right by some categories, as we will see when we break the distribution into each of the nine primaries.

In [ ]:
data_edited.price_usd.describe()

In [ ]:
sns.displot(data_edited,
            x = "price_usd",
            hue = "primary_category",
            col = "primary_category")

- Fragrance: Bulk of data spread between 25.00 USD and 150.00 USD, with outliers reaching up to 400.00 USD
- Bath and Body: Distribution contained within 0 - 50.00 USD, with some outliers reaching towards 100.00 USD.
- Mini Size: All data (with the exception of a few outliers) contained within 0 - 50.00 USD. Outliers do not exceed 75.00 USD.
- Hair: Distribution skewed to the right, with center around 50.00 USD and outliers reaching up to 300.00 USD.
- Makeup: Narrow distribution centering around 30 - 50.00 USD. Outliers reaching up to 150.00 USD.
- Skincare: Right skewing distribution with a wider center plateuing around 50.00 USD. Noticeably more data points in above 50.00 USD range.
- Tools and Brushes: Prices range from 0 to around 35.00 USD. Few outliers near 50.00 USD.
- Men: Prices range from 20.00 USD to just under 50.00 USD. Very few data points.
- Gifts: One entry at 50.00 USD.

### Key Takeaways and What's Next <a id='key_takeaways_and_whats_next'></a>

In this project, I learned a plethora of new information on Python, ranging from data cleaning and transformation to Natural Language Processing and visualization (and of course, lots of debugging). I was admittedly relunctant to learn Python for some time; there were even numerous times in the project where I knew I could complete the tasks in SQL or Power BI. However, I'm glad that I stuck with Python for the entirety of the project, as leveraging the language has given me the familiarity needed to utilize it as a tool in the future.

Throughout the project I also found myself frequently taking "deep dives" into concepts that were better suited for more advanced analyses, such as feature extraction and NLP toolkits. Though these were ultimately out of scope for this project, I would love to see how I can incorporate more challenging concepts like these in the future for a more rewarding analysis. 